# 5.1 端侧NPU适配

端侧NPU（Neural Processing Unit）是专为神经网络推理设计的加速器，具有高算力、低功耗的特点。不同厂商的NPU架构和SDK不同，需要针对性适配。

## 什么是NPU适配？

NPU适配是将训练好的模型转换并优化以在特定NPU上高效运行的过程，核心工作包括：
- **算子映射**：将模型中的算子映射到NPU支持的硬件指令
- **量化支持**：NPU通常仅支持INT8/INT4精度，需要将FP32模型量化
- **内存布局优化**：NPU对张量排布有特定要求（如NCHW/NHWC）
- **计算图优化**：针对NPU的并行计算单元进行算子融合和调度优化

## 为什么需要NPU适配？

1. **架构差异大**：不同厂商NPU（华为Ascend、高通Hexagon、苹果Neural Engine、联发科APU）的指令集、内存层次、并行度完全不同
2. **功耗效率**：NPU的能效比（TOPS/W）远超CPU和GPU，是端侧AI推理的首选
3. **算子限制**：NPU支持的算子集合通常小于GPU，复杂算子需要拆分或回退到CPU

## NPU适配的数学原理

量化误差的MSE上界：$E[|x - \hat{x}|^2] \leq \frac{s^2}{12}$，其中$s$为量化步长
Attention算子拆分：$\text{softmax}(QK^T/\sqrt{d})V = \text{softmax}(QK^T/\sqrt{d}) \cdot V$，可拆分为矩阵乘+softmax+矩阵乘三步执行

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

torch.manual_seed(42)
print(f"PyTorch version: {torch.__version__}")

---
## 5.1.1 NPU架构深潜

理解NPU的硬件架构是高效适配的基础。不同厂商NPU的内部结构差异巨大，直接影响算子映射策略和性能优化方向。

### NPU核心架构组件

典型NPU由以下核心组件构成：

| 组件 | 功能 | 典型规格 |
|------|------|----------|
| **MAC阵列** | 矩阵乘加速单元，NPU的核心计算引擎 | 256-4096个MAC单元，支持INT8/INT4 |
| **片上SRAM** | 高速片上缓存，存储激活值和部分权重 | 256KB-4MB，带宽>1TB/s |
| **DMA引擎** | 在DRAM和SRAM之间搬运数据 | 多通道并行，支持2D/3D传输 |
| **标量处理器** | 执行标量运算（softmax、LayerNorm等） | 1-4核，FP16/INT32 |
| **向量处理器** | 执行向量运算（激活函数、逐元素运算） | 128-512位宽，FP16/INT8/INT4 |
| **指令控制器** | 解析和调度NPU指令，管理流水线 | 支持指令级并行 |

### 主流NPU架构对比

| 特性 | 高通Hexagon | 苹果Neural Engine | 华为昇腾 | 联发科APU |
|------|------------|------------------|---------|----------|
| **计算范式** | V68 ISA, HVX+HTP | 数据流架构 | 达芬奇架构, 3D Cube | Cadence DSP+自研加速 |
| **MAC阵列** | 4x128 INT8 MAC | 16x128 INT8 MAC | 16x16x16 Cube | 可配置MAC阵列 |
| **片上SRAM** | 4MB (8 Gen3) | ~8MB (M4) | 2-8MB (310P) | 2-4MB |
| **INT4支持** | 原生支持 | 不支持 | 部分支持 | 不支持 |
| **动态shape** | 有限支持 | 不支持 | 有限支持 | 不支持 |
| **峰值算力** | 75 TOPS (8 Elite) | 38 TOPS (M4) | 128 TOPS (310P) | 45 TOPS (9400) |
| **能效比** | ~15 TOPS/W | ~12 TOPS/W | ~10 TOPS/W | ~10 TOPS/W |

### NPU内存层次与数据流

```
┌─────────────────────────────────────────────────┐
│                   DRAM (4-16GB)                  │  带宽: 30-60 GB/s
│            模型权重 / KV Cache / 激活值           │  延迟: ~100ns
└──────────────────────┬──────────────────────────┘
                       │ DMA传输 (异步, 多通道)
┌──────────────────────▼──────────────────────────┐
│              片上SRAM (256KB-8MB)                │  带宽: >1 TB/s
│         当前层权重 / 当前tile激活值               │  延迟: ~1ns
└──────────────────────┬──────────────────────────┘
                       │ 寄存器文件
┌──────────────────────▼──────────────────────────┐
│               MAC阵列 / 向量单元                  │  带宽: >10 TB/s
│              矩阵乘 / 逐元素运算                  │  延迟: <1ns
└─────────────────────────────────────────────────┘
```

**关键洞察**：NPU性能的瓶颈通常不在MAC阵列的计算能力，而在DRAM→SRAM的数据搬运。因此，**最大化SRAM复用、最小化DRAM访问次数**是NPU优化的核心原则。

In [ ]:
@dataclass
class NPUArchSpec:
    name: str
    mac_array: Tuple[int, int]
    sram_kb: int
    dram_bandwidth_gbs: float
    sram_bandwidth_gbs: float
    supported_precisions: List[str]
    peak_tops: float
    tdp_watts: float

    @property
    def tops_per_watt(self):
        return self.peak_tops / self.tdp_watts

    @property
    def sram_capacity_mb(self):
        return self.sram_kb / 1024

    def mac_utilization_for_gemm(self, M, N, K, dtype_bytes=1):
        """估算GEMM在MAC阵列上的利用率"""
        mac_rows, mac_cols = self.mac_array
        row_util = min(M / mac_rows, 1.0) if M >= mac_rows else M / mac_rows
        col_util = min(N / mac_cols, 1.0) if N >= mac_cols else N / mac_cols
        return row_util * col_util

NPU_SPECS = {
    'qualcomm_hexagon_8elite': NPUArchSpec(
        name='Qualcomm Hexagon (8 Elite)',
        mac_array=(4, 128), sram_kb=4096,
        dram_bandwidth_gbs=60, sram_bandwidth_gbs=1200,
        supported_precisions=['int8', 'int4', 'fp16'],
        peak_tops=75, tdp_watts=5.0
    ),
    'apple_ane_m4': NPUArchSpec(
        name='Apple Neural Engine (M4)',
        mac_array=(16, 128), sram_kb=8192,
        dram_bandwidth_gbs=120, sram_bandwidth_gbs=2000,
        supported_precisions=['fp16', 'int8'],
        peak_tops=38, tdp_watts=3.0
    ),
    'huawei_ascend_310p': NPUArchSpec(
        name='Huawei Ascend 310P',
        mac_array=(16, 16), sram_kb=8192,
        dram_bandwidth_gbs=51.2, sram_bandwidth_gbs=1000,
        supported_precisions=['int8', 'int4', 'fp16'],
        peak_tops=128, tdp_watts=12.0
    ),
    'mediatek_apu_9400': NPUArchSpec(
        name='MediaTek APU (9400)',
        mac_array=(8, 64), sram_kb=4096,
        dram_bandwidth_gbs=50, sram_bandwidth_gbs=800,
        supported_precisions=['int8', 'int16', 'fp16'],
        peak_tops=45, tdp_watts=4.5
    ),
}

print("=== 主流端侧NPU架构规格对比 ===")
print(f"{'NPU':<30} {'MAC阵列':<12} {'SRAM':<10} {'DRAM BW':<10} {'峰值算力':<12} {'能效比':<10}")
print(f"{'':30} {'':12} {'(MB)':10} {'(GB/s)':10} {'(TOPS)':12} {'(TOPS/W)':10}")
print("-" * 84)
for key, spec in NPU_SPECS.items():
    print(f"{spec.name:<30} {str(spec.mac_array):<12} {spec.sram_capacity_mb:<10.1f} "
          f"{spec.dram_bandwidth_gbs:<10.1f} {spec.peak_tops:<12.0f} {spec.tops_per_watt:<10.1f}")

print(f"\n=== GEMM MAC利用率分析 (7B模型, hidden_dim=4096) ===")
M, N, K = 1, 4096, 4096
for key, spec in NPU_SPECS.items():
    util = spec.mac_utilization_for_gemm(M, N, K)
    print(f"  {spec.name}: MAC利用率 = {util:.4%} (M={M}, MAC行={spec.mac_array[0]})")

print(f"\n关键洞察:")
print(f"  1. NPU的MAC阵列为矩阵乘优化，decode阶段batch=1时严重欠利用")
print(f"  2. SRAM容量决定单次可处理的tile大小，影响算子切分策略")
print(f"  3. DRAM带宽是decode阶段的实际瓶颈，量化减少搬运量最有效")
print(f"  4. 苹果ANE的MAC行数最多(16行)，batch=1时利用率相对较高")
print(f"  5. 实际产业中常通过batching、投机解码来提高MAC利用率")

---
## 5.1.2 NPU算子兼容性与分解策略

### 算子兼容性分析

不同NPU支持的算子集合不同，需要检查模型中的算子是否被目标NPU支持，不支持的算子需要回退到CPU执行。

### CPU回退的代价

1. **CPU执行延迟**：CPU执行单个算子的延迟可能是NPU的2-5倍
2. **数据搬运开销**：NPU和CPU之间的数据传输延迟约$10-100\mu s$
3. **端到端延迟**：即使只有1-2个算子回退，也可能成为整体推理的瓶颈

回退延迟模型：$T_{\text{fallback}} = T_{\text{NPU}\to\text{CPU}} + T_{\text{CPU}} + T_{\text{CPU}\to\text{NPU}}$

### LLM关键算子的NPU分解策略

LLM中的关键算子（Attention、RoPE、SwiGLU等）通常不被NPU原生支持，需要分解为NPU支持的基础算子组合：

| LLM算子 | NPU分解策略 | 分解后算子 |
|---------|------------|-----------|
| **Multi-Head Attention** | 拆分为QKV投影+注意力计算+输出投影 | MatMul+Softmax+MatMul+Concat |
| **RoPE** | 预计算cos/sin表，分解为逐元素乘加 | Elementwise Mul+Add |
| **SwiGLU** | 拆分为两个线性层+SiLU+逐元素乘 | MatMul+SiLU+Elementwise Mul |
| **RMSNorm** | 分解为平方+求和+开方+归一化 | Pow+Reduce+Div+Mul |
| **KV Cache更新** | 拆分为索引写入+读取 | Scatter+Gather |
| **Top-K Sampling** | 回退CPU（NPU不支持动态排序） | CPU回退 |
| **Token Embedding** | 查表操作，NPU支持 | Gather（部分NPU支持） |

In [ ]:
@dataclass
class OpDecomposition:
    original_op: str
    decomposed_ops: List[str]
    npu_executable: bool
    estimated_overhead: float
    notes: str

class NPUOperatorAnalyzer:
    """NPU算子兼容性分析与分解策略"""
    def __init__(self):
        self.npu_support = {
            'qualcomm_hexagon': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'silu', 'layer_norm',
                              'batch_norm', 'softmax', 'add', 'mul', 'matmul',
                              'reshape', 'transpose', 'concat', 'split', 'gather',
                              'quantize', 'dequantize', 'elementwise_mul', 'elementwise_add',
                              'reduce_sum', 'pow', 'sqrt', 'div', 'scatter'],
                'unsupported': ['attention_fused', 'rope_fused', 'topk', 'multinomial',
                                'dynamic_shape_ops', 'sort', 'unique'],
                'precision': ['int8', 'int4', 'fp16'],
                'tops': 75,
            },
            'apple_ane': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'silu', 'layer_norm',
                              'batch_norm', 'softmax', 'add', 'mul', 'matmul',
                              'reshape', 'transpose', 'concat', 'gather',
                              'elementwise_mul', 'elementwise_add', 'reduce_sum'],
                'unsupported': ['attention_fused', 'rope_fused', 'topk', 'quantize_int4',
                                'scatter', 'dynamic_shape_ops', 'sort', 'pow'],
                'precision': ['fp16', 'int8'],
                'tops': 38,
            },
            'mediatek_apu': {
                'supported': ['conv2d', 'linear', 'relu', 'gelu', 'silu', 'layer_norm',
                              'softmax', 'add', 'mul', 'matmul', 'concat', 'gather',
                              'elementwise_mul', 'elementwise_add'],
                'unsupported': ['attention_fused', 'rope_fused', 'topk', 'multinomial',
                                'int4', 'scatter', 'dynamic_shape_ops', 'pow', 'sort'],
                'precision': ['int8', 'int16', 'fp16'],
                'tops': 45,
            },
        }

        self.llm_decompositions = {
            'attention_fused': OpDecomposition(
                original_op='Multi-Head Attention',
                decomposed_ops=['matmul(QKV_proj)', 'reshape+transpose', 'matmul(Q·K^T)',
                                'softmax', 'matmul(attn·V)', 'reshape+transpose', 'matmul(out_proj)'],
                npu_executable=True,
                estimated_overhead=1.15,
                notes='分解后7个子算子均可NPU执行，但中间结果需SRAM暂存'
            ),
            'rope_fused': OpDecomposition(
                original_op='Rotary Position Embedding',
                decomposed_ops=['gather(cos/sin_table)', 'elementwise_mul', 'elementwise_add'],
                npu_executable=True,
                estimated_overhead=1.05,
                notes='预计算cos/sin表作为常量嵌入模型，推理时仅需逐元素乘加'
            ),
            'swiglu': OpDecomposition(
                original_op='SwiGLU Activation',
                decomposed_ops=['matmul(gate_proj)', 'silu', 'matmul(up_proj)',
                                'elementwise_mul', 'matmul(down_proj)'],
                npu_executable=True,
                estimated_overhead=1.0,
                notes='分解后全部可NPU执行，SiLU在大多数NPU有硬件支持'
            ),
            'rmsnorm': OpDecomposition(
                original_op='RMS Normalization',
                decomposed_ops=['pow(x,2)', 'reduce_sum', 'mean', 'add(eps)',
                                'sqrt', 'div', 'elementwise_mul(gamma)'],
                npu_executable=True,
                estimated_overhead=1.10,
                notes='pow+sqrt部分NPU不支持FP16，可能需回退标量处理器'
            ),
            'topk': OpDecomposition(
                original_op='Top-K Sampling',
                decomposed_ops=['sort', 'gather'],
                npu_executable=False,
                estimated_overhead=3.0,
                notes='排序操作NPU不支持，必须回退CPU；但仅decode最后一步，影响有限'
            ),
        }

    def analyze_model(self, target_npu: str) -> Dict:
        """分析LLM模型在目标NPU上的算子兼容性"""
        npu_info = self.npu_support[target_npu]
        results = {'fully_supported': [], 'decomposable': [], 'cpu_fallback': []}

        for op_name, decomp in self.llm_decompositions.items():
            all_decomposed_supported = all(
                sub_op in npu_info['supported'] or
                any(kw in sub_op for kw in npu_info['supported'])
                for sub_op in decomp.decomposed_ops
            )
            if op_name in npu_info['supported']:
                results['fully_supported'].append(decomp)
            elif decomp.npu_executable and all_decomposed_supported:
                results['decomposable'].append(decomp)
            else:
                results['cpu_fallback'].append(decomp)

        return results

analyzer = NPUOperatorAnalyzer()

for npu_name in ['qualcomm_hexagon', 'apple_ane', 'mediatek_apu']:
    info = analyzer.npu_support[npu_name]
    result = analyzer.analyze_model(npu_name)
    print(f"\n=== {npu_name} ({info['tops']} TOPS) ===")
    print(f"支持精度: {info['precision']}")
    print(f"\n  可分解到NPU执行的算子:")
    for decomp in result['decomposable']:
        print(f"    {decomp.original_op}")
        print(f"      → {' → '.join(decomp.decomposed_ops)}")
        print(f"      开销系数: {decomp.estimated_overhead:.2f}x | {decomp.notes}")
    print(f"\n  需CPU回退的算子:")
    for decomp in result['cpu_fallback']:
        print(f"    {decomp.original_op}")
        print(f"      → 回退代价: {decomp.estimated_overhead:.2f}x | {decomp.notes}")

print(f"\n=== 产业实践要点 ===")
print(f"1. Attention分解是NPU适配的核心工作，需仔细优化中间结果的SRAM布局")
print(f"2. RoPE预计算表方案比在线计算更高效，但增加了模型体积")
print(f"3. Top-K/Sampling回退CPU不可避免，但仅占decode总延迟的5-10%")
print(f"4. 动态shape是最大挑战：NPU编译期需固定shape，运行时需padding+mask")

---
## 5.1.3 Attention算子NPU分解的详细实现

Attention是LLM的核心算子，也是NPU适配最复杂的部分。下面展示如何将融合Attention分解为NPU可执行的基础算子序列，并分析SRAM使用和性能。

### 分解策略

标准Multi-Head Attention：
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

NPU分解为7步：
1. **QKV投影**：$Q = xW_Q, K = xW_K, V = xW_V$ → 3个MatMul
2. **RoPE**：$Q = Q \cdot \cos\theta + Q_{\text{rot}} \cdot \sin\theta$ → 逐元素运算
3. **注意力分数**：$S = QK^T / \sqrt{d_k}$ → MatMul + 标量乘
4. **Softmax**：$A = \text{softmax}(S)$ → Softmax（NPU支持）
5. **注意力输出**：$O = AV$ → MatMul
6. **头拼接**：reshape + transpose → 内存布局变换
7. **输出投影**：$Y = OW_O$ → MatMul

### SRAM使用分析

分解后的关键约束是SRAM容量。假设SRAM容量为$S$字节：
- QKV投影：需存储$Q, K, V$ → $3 \times B \times L \times d$ 字节
- 注意力分数：$B \times H \times L \times L$ 字节（$L^2$是内存瓶颈）
- **Tiling策略**：当$L$较大时，需将$Q$和$K$沿$L$维度分tile计算

In [ ]:
class NPUAttentionDecomposer:
    """Attention算子NPU分解与SRAM规划"""
    def __init__(self, sram_capacity_kb=4096, dtype_bytes=2):
        self.sram_capacity = sram_capacity_kb * 1024
        self.dtype_bytes = dtype_bytes

    def compute_sram_usage(self, batch, seq_len, hidden_dim, n_heads, phase='prefill'):
        """计算各分解步骤的SRAM使用量"""
        head_dim = hidden_dim // n_heads
        steps = {}

        qkv_size = 3 * batch * seq_len * hidden_dim * self.dtype_bytes
        steps['1_qkv_proj'] = {
            'tensors': ['Q', 'K', 'V'],
            'size_bytes': qkv_size,
            'size_mb': qkv_size / 1024 / 1024,
        }

        attn_score_size = batch * n_heads * seq_len * seq_len * self.dtype_bytes
        steps['2_attn_score'] = {
            'tensors': ['attn_scores'],
            'size_bytes': attn_score_size,
            'size_mb': attn_score_size / 1024 / 1024,
        }

        attn_output_size = batch * n_heads * seq_len * head_dim * self.dtype_bytes
        steps['3_attn_output'] = {
            'tensors': ['attn_output'],
            'size_bytes': attn_output_size,
            'size_mb': attn_output_size / 1024 / 1024,
        }

        total_peak = qkv_size + attn_score_size + attn_output_size
        steps['total_peak'] = {
            'tensors': ['peak (no tiling)'],
            'size_bytes': total_peak,
            'size_mb': total_peak / 1024 / 1024,
        }

        return steps

    def compute_tiled_sram(self, batch, seq_len, hidden_dim, n_heads, tile_size=None):
        """计算Tiling策略下的SRAM使用量"""
        head_dim = hidden_dim // n_heads
        if tile_size is None:
            tile_size = seq_len

        q_tile = batch * tile_size * hidden_dim * self.dtype_bytes
        kv_full = 2 * batch * seq_len * hidden_dim * self.dtype_bytes
        score_tile = batch * n_heads * tile_size * seq_len * self.dtype_bytes
        out_tile = batch * tile_size * hidden_dim * self.dtype_bytes

        tiled_peak = q_tile + kv_full + score_tile + out_tile
        return {
            'tile_size': tile_size,
            'n_tiles': (seq_len + tile_size - 1) // tile_size,
            'peak_sram_mb': tiled_peak / 1024 / 1024,
            'fits_sram': tiled_peak <= self.sram_capacity,
        }

    def find_optimal_tile_size(self, batch, seq_len, hidden_dim, n_heads):
        """搜索满足SRAM约束的最大tile_size"""
        for tile_size in [seq_len, 512, 256, 128, 64, 32]:
            result = self.compute_tiled_sram(batch, seq_len, hidden_dim, n_heads, tile_size)
            if result['fits_sram']:
                return result
        return self.compute_tiled_sram(batch, seq_len, hidden_dim, n_heads, 16)

decomposer = NPUAttentionDecomposer(sram_capacity_kb=4096, dtype_bytes=2)

print("=== Attention分解的SRAM使用分析 ===")
print(f"假设: SRAM容量 = 4MB, dtype = FP16")

for model_name, dim, heads in [('7B (4096)', 4096, 32), ('1.5B (2048)', 2048, 16)]:
    print(f"\n--- {model_name} ---")
    for seq_len in [128, 512, 2048]:
        steps = decomposer.compute_sram_usage(1, seq_len, dim, heads)
        peak_mb = steps['total_peak']['size_mb']
        fits = "✓" if steps['total_peak']['size_bytes'] <= decomposer.sram_capacity else "✗"
        print(f"  seq_len={seq_len}: 峰值SRAM={peak_mb:.2f}MB {fits}")

        if steps['total_peak']['size_bytes'] > decomposer.sram_capacity:
            tiled = decomposer.find_optimal_tile_size(1, seq_len, dim, heads)
            print(f"    → Tiling: tile_size={tiled['tile_size']}, n_tiles={tiled['n_tiles']}, "
                  f"峰值SRAM={tiled['peak_sram_mb']:.2f}MB")

print(f"\n=== Tiling策略的关键洞察 ===")
print(f"1. prefill阶段seq_len大时，注意力分数矩阵(L×L)是SRAM瓶颈")
print(f"2. Tiling将L×L矩阵拆分为tile×L，SRAM从O(L²)降至O(tile×L)")
print(f"3. decode阶段seq_len=1，无需tiling，SRAM压力主要来自KV Cache")
print(f"4. 实际NPU SDK（QNN/CANN）会自动执行tiling，但理解原理有助于调试")

---
## 5.1.4 NPU量化适配

将FP32模型量化为NPU支持的低精度格式（INT8/INT4），并确保量化后精度满足要求。

### NPU量化适配的核心挑战

1. **NPU仅支持低精度**：大多数端侧NPU仅支持INT8/INT4运算
2. **精度-效率权衡**：INT8几乎无损但加速有限，INT4加速更大但精度损失更多
3. **混合精度配置**：不同层对量化的敏感度不同，需要逐层配置最优精度
4. **NPU精度约束**：不同NPU支持的精度组合不同，需在硬件约束下搜索最优配置

### 混合精度量化配置优化

$$\min_{(W_b, A_b)} \text{Size}(W_b) \quad \text{s.t.} \quad \text{Accuracy}(W_b, A_b) \geq \text{threshold} \quad \text{and} \quad (W_b, A_b) \in \text{NPU\_Supported}$$

### 各NPU精度支持详情

| NPU | 权重精度 | 激活精度 | MAC吞吐量 | 注意事项 |
|-----|---------|---------|----------|----------|
| 高通Hexagon (8 Elite) | INT8/INT4 | INT8/FP16 | 75 TOPS (INT8) | INT4需QNN 2.20+, 仅限权重 |
| 苹果Neural Engine (M4) | FP16/INT8 | FP16 | 38 TOPS (FP16) | ANE对FP16优化最佳，INT8需coremltools转换 |
| 华为昇腾 (310P) | INT8/INT4 | INT8/INT16 | 128 TOPS (INT8) | INT4通过OM模型配置，需CANN 7.0+ |
| 联发科APU (9400) | INT8/INT16 | INT8/FP16 | 45 TOPS (INT8) | 无INT4支持，INT8为主要推理精度 |

In [ ]:
@dataclass
class QuantConfig:
    weight_prec: str
    activation_prec: str
    estimated_size_mb: float
    fits_budget: bool
    npu_supported: bool
    expected_ppl_increase: float
    decode_speedup_vs_fp16: float

class NPUQuantizationAdapter:
    """NPU量化适配器: 根据硬件约束和精度需求推荐最优量化配置"""
    PRECISION_SIZE = {'int4': 0.5, 'int8': 1.0, 'int16': 2.0, 'fp16': 2.0, 'fp32': 4.0}
    PPL_ESTIMATE = {
        ('int4', 'int8'): 0.3, ('int4', 'fp16'): 0.2,
        ('int8', 'int8'): 0.05, ('int8', 'fp16'): 0.03,
        ('fp16', 'fp16'): 0.0,
    }
    DECODE_SPEEDUP = {
        ('int4', 'int8'): 3.5, ('int4', 'fp16'): 3.0,
        ('int8', 'int8'): 1.8, ('int8', 'fp16'): 1.5,
        ('fp16', 'fp16'): 1.0,
    }

    def __init__(self, target_npu='qualcomm_hexagon'):
        self.target_npu = target_npu
        self.supported_precisions = {
            'qualcomm_hexagon': {
                'weight': ['int4', 'int8', 'fp16'],
                'activation': ['int8', 'fp16'],
            },
            'apple_ane': {
                'weight': ['int8', 'fp16'],
                'activation': ['fp16'],
            },
            'mediatek_apu': {
                'weight': ['int8', 'int16'],
                'activation': ['int8', 'fp16'],
            },
        }

    def get_optimal_config(self, model_size_mb, memory_budget_mb, max_ppl_increase=0.5):
        """根据模型大小、内存预算和精度约束推荐量化配置"""
        precisions = self.supported_precisions[self.target_npu]
        configs = []

        for w_prec in precisions['weight']:
            for a_prec in precisions['activation']:
                w_ratio = self.PRECISION_SIZE[w_prec] / self.PRECISION_SIZE['fp32']
                estimated_size = model_size_mb * w_ratio
                ppl_inc = self.PPL_ESTIMATE.get((w_prec, a_prec), 1.0)
                speedup = self.DECODE_SPEEDUP.get((w_prec, a_prec), 1.0)

                configs.append(QuantConfig(
                    weight_prec=w_prec, activation_prec=a_prec,
                    estimated_size_mb=estimated_size,
                    fits_budget=estimated_size <= memory_budget_mb,
                    npu_supported=True,
                    expected_ppl_increase=ppl_inc,
                    decode_speedup_vs_fp16=speedup,
                ))

        valid_configs = [c for c in configs
                         if c.fits_budget and c.expected_ppl_increase <= max_ppl_increase]
        if not valid_configs:
            valid_configs = sorted(configs, key=lambda x: x.estimated_size_mb)

        return sorted(valid_configs, key=lambda x: -x.decode_speedup_vs_fp16)

print("=== NPU量化配置推荐 (7B模型, 4GB FP32) ===")
for npu_name in ['qualcomm_hexagon', 'apple_ane', 'mediatek_apu']:
    adapter = NPUQuantizationAdapter(npu_name)
    configs = adapter.get_optimal_config(model_size_mb=14000, memory_budget_mb=4000, max_ppl_increase=0.5)
    print(f"\n--- {npu_name} ---")
    print(f"{'权重':<8} {'激活':<8} {'大小(MB)':<10} {'适合内存':<8} {'PPL增加':<10} {'Decode加速':<10}")
    print("-" * 54)
    for cfg in configs:
        print(f"{cfg.weight_prec:<8} {cfg.activation_prec:<8} {cfg.estimated_size_mb:<10.0f} "
              f"{'✓' if cfg.fits_budget else '✗':<8} {cfg.expected_ppl_increase:<10.2f} {cfg.decode_speedup_vs_fp16:<10.1f}x")

print(f"\n=== 产业级量化配置决策流程 ===")
print(f"1. 确定目标NPU支持的精度组合（硬件约束）")
print(f"2. 根据内存预算筛选可用的权重精度（存储约束）")
print(f"3. 根据精度要求筛选可接受的配置（质量约束）")
print(f"4. 在满足所有约束的配置中选择decode加速比最高的（性能最优）")
print(f"5. 对敏感层（首尾层、embedding层）保持更高精度（混合精度微调）")

---
## 5.1.5 NPU部署管线

产业级NPU部署不是简单的"模型转换→推理"，而是一个包含多阶段优化和验证的完整管线。

### 通用NPU部署管线

```
训练模型 (PyTorch)
    │
    ▼
① 模型导出 → ONNX / TorchScript / 自定义IR
    │
    ▼
② 算子兼容性检查 → 不支持的算子分解/替换/回退
    │
    ▼
③ 计算图优化 → 算子融合 + 常量折叠 + 死代码消除
    │
    ▼
④ 量化 → PTQ/QAT → INT8/INT4
    │
    ▼
⑤ NPU编译 → 针对目标NPU生成优化指令
    │
    ▼
⑥ 精度验证 → 逐层对比 + 端到端精度测试
    │
    ▼
⑦ 性能Profile → 算子级耗时 + 内存占用 + 功耗
    │
    ▼
⑧ 部署打包 → 生成二进制 + 运行时配置
```

### 各厂商SDK的部署管线对比

| 阶段 | 高通QNN | 苹果Core ML | 华为CANN | 联发科NeuroPilot |
|------|---------|------------|---------|-----------------|
| **模型导入** | ONNX→QNN IR | coremltools转换 | ONNX→OM | TFLite/ONNX |
| **量化工具** | QNN Quantizer | coremltools量化 | AMCT量化工具 | NeuroPilot Quantizer |
| **编译优化** | QNN Graph Optimizer | Core ML Compiler | ACE编译器 | NeuroPilot Compiler |
| **NPU编译** | QNN Context Binary生成 | .mlmodelc/.mlpackage | OM模型文件 | NeuroPilot Binary |
| **运行时** | QNN Runtime | Core ML Framework | ACL推理引擎 | NeuroPilot Runtime |
| **Profile工具** | QNN Profiler | Instruments | msprof | NeuroPilot Profiler |
| **调试工具** | QNN Debug | Core ML Debugger | 对比工具 | 日志系统 |

In [ ]:
class NPUDeploymentPipeline:
    """NPU部署管线模拟器"""
    def __init__(self, vendor: str):
        self.vendor = vendor
        self.pipeline_stages = {
            'qualcomm': [
                {'name': '模型导出', 'tool': 'torch.onnx.export', 'output': 'model.onnx',
                 'cmd': 'python -m tf2onnx.convert --tflite model.tflite --output model.onnx'},
                {'name': 'QNN转换', 'tool': 'qnn-onnx-converter', 'output': 'model.cpp',
                 'cmd': 'qnn-onnx-converter --input_model model.onnx --output model.cpp'},
                {'name': 'QNN量化', 'tool': 'qnn-post-training-quantization', 'output': 'quantized.cpp',
                 'cmd': 'qnn-post-training-quantization --input_model model.cpp --bias_dtype int32 --act_dtype int8 --weight_dtype int8'},
                {'name': 'QNN编译', 'tool': 'qnn-model-lib-generator', 'output': 'libQnnModel.so + .bin',
                 'cmd': 'qnn-model-lib-generator --model quantized.cpp --output_dir ./output'},
                {'name': 'Context Binary', 'tool': 'qnn-context-binary-generator', 'output': 'model.bin',
                 'cmd': 'qnn-context-binary-generator --backend libQnnHtp.so --model ./output/libQnnModel.so --output model.bin'},
                {'name': '端侧推理', 'tool': 'QNN Runtime', 'output': 'inference results',
                 'cmd': 'qnn-net-run --model model.bin --input input_list.txt'},
            ],
            'apple': [
                {'name': '模型导出', 'tool': 'torch.export + coremltools', 'output': 'model.mlpackage',
                 'cmd': 'coremltools.convert(model, inputs=[...])'},
                {'name': 'ANE优化', 'tool': 'coremltools.optimize', 'output': 'optimized.mlpackage',
                 'cmd': 'ct.optimize.coreml.linear_quantize_weights(model)'},
                {'name': '编译部署', 'tool': 'Xcode', 'output': 'App Bundle',
                 'cmd': 'Xcode自动编译.mlpackage到.mlmodelc'},
                {'name': '端侧推理', 'tool': 'Core ML Framework', 'output': 'inference results',
                 'cmd': 'model.predict(input_data)'},
            ],
            'huawei': [
                {'name': '模型导出', 'tool': 'torch.onnx.export', 'output': 'model.onnx',
                 'cmd': 'torch.onnx.export(model, dummy_input, "model.onnx")'},
                {'name': 'AMCT量化', 'tool': 'amct_onnx', 'output': 'quantized_model.onnx',
                 'cmd': 'amct_onnx.create_quant_config + calibrate + quantize'},
                {'name': 'ACE编译', 'tool': 'atc (Ascend Tensor Compiler)', 'output': 'model.om',
                 'cmd': 'atc --model=quantized_model.onnx --framework=5 --output=model.om --soc_version=Ascend310P3'},
                {'name': '端侧推理', 'tool': 'ACL (Ascend Computing Language)', 'output': 'inference results',
                 'cmd': 'acl.mdl.execute(model_id, inputs, outputs)'},
            ],
        }

    def print_pipeline(self):
        stages = self.pipeline_stages.get(self.vendor, [])
        print(f"\n=== {self.vendor.upper()} NPU部署管线 ===")
        for i, stage in enumerate(stages, 1):
            print(f"\n  ①{stage['name']}")
            print(f"    工具: {stage['tool']}")
            print(f"    输出: {stage['output']}")
            print(f"    命令: {stage['cmd']}")
            if i < len(stages):
                print(f"    ↓")

    def estimate_deployment_time(self, model_params_b: float) -> Dict:
        """估算各阶段耗时"""
        base = model_params_b
        estimates = {
            'export': base * 2,
            'convert': base * 5,
            'quantize': base * 15,
            'compile': base * 30,
            'verify': base * 10,
        }
        return estimates

for vendor in ['qualcomm', 'apple', 'huawei']:
    pipeline = NPUDeploymentPipeline(vendor)
    pipeline.print_pipeline()

print(f"\n=== 7B模型NPU部署耗时估算 ===")
pipeline = NPUDeploymentPipeline('qualcomm')
est = pipeline.estimate_deployment_time(7)
total = sum(est.values())
for stage, time_min in est.items():
    print(f"  {stage}: ~{time_min:.0f}分钟")
print(f"  总计: ~{total:.0f}分钟 ({total/60:.1f}小时)")
print(f"\n注意: 编译阶段耗时最长，NPU编译器需搜索最优算子实现和调度策略")
print(f"建议: 开发阶段使用小模型快速迭代，最终用完整模型编译部署")

---
## 5.1.6 动态Shape处理与内存管理

LLM推理的序列长度动态变化，但NPU通常只支持静态shape，这是NPU适配的核心难题之一。

### 动态Shape的挑战

| 场景 | Shape变化 | NPU处理难度 |
|------|----------|------------|
| **Prefill** | seq_len随输入变化 | 高 - 需编译多个shape或padding |
| **Decode** | KV Cache长度逐步增长 | 高 - 每步shape不同 |
| **Batch** | 请求数动态变化 | 中 - 通常固定batch=1 |
| **多模态** | 图像token数变化 | 高 - 需编译多个变体 |

### 解决方案

1. **Padding + Mask**：将所有输入padding到固定长度（如max_seq_len），用attention mask忽略padding部分
   - 优点：编译一次，运行时零开销
   - 缺点：短序列浪费算力，长序列受max_seq_len限制

2. **多shape编译**：为常用shape分别编译（如seq_len=128, 256, 512, 1024），运行时选择最接近的
   - 优点：各shape均有优化
   - 缺点：编译时间长，存储多个二进制

3. **Dynamic Batch Dispatch**：将prefill和decode分别编译，decode固定batch=1+seq_len=1
   - 优点：最常用的decode路径高度优化
   - 缺点：prefill仍需处理动态shape

### KV Cache内存管理

NPU上的KV Cache管理比GPU更受限：
- **预分配**：NPU不支持动态内存分配，需在编译期预分配最大KV Cache空间
- **PagedAttention限制**：NPU不支持非连续内存访问，PagedAttention难以直接映射
- **实际方案**：预分配固定大小的KV Cache buffer，用偏移量管理写入位置

In [ ]:
class NPUMemoryPlanner:
    """NPU内存规划器 - 在编译期规划推理时的内存分配"""
    def __init__(self, total_memory_mb, sram_kb=4096):
        self.total_memory = total_memory_mb * 1024 * 1024
        self.sram = sram_kb * 1024

    def plan_llm_memory(self, n_layers, hidden_dim, n_heads, max_seq_len,
                        weight_dtype_bytes=1, kv_dtype_bytes=1, batch_size=1):
        """规划LLM推理的内存分配"""
        head_dim = hidden_dim // n_heads

        weight_size = n_layers * (
            hidden_dim * hidden_dim * 4 * weight_dtype_bytes +
            hidden_dim * hidden_dim * 3 * weight_dtype_bytes +
            hidden_dim * 2 * weight_dtype_bytes
        )

        kv_cache_size = 2 * n_layers * batch_size * max_seq_len * hidden_dim * kv_dtype_bytes

        activation_size = batch_size * max_seq_len * hidden_dim * 2 * 4

        total = weight_size + kv_cache_size + activation_size

        return {
            'weights_mb': weight_size / 1024 / 1024,
            'kv_cache_mb': kv_cache_size / 1024 / 1024,
            'activations_mb': activation_size / 1024 / 1024,
            'total_mb': total / 1024 / 1024,
            'budget_mb': self.total_memory / 1024 / 1024,
            'fits': total <= self.total_memory,
            'max_seq_len': max_seq_len,
        }

    def find_max_seq_len(self, n_layers, hidden_dim, n_heads,
                         weight_dtype_bytes=1, kv_dtype_bytes=1, batch_size=1):
        """搜索满足内存预算的最大序列长度"""
        for seq_len in [4096, 2048, 1024, 512, 256, 128]:
            plan = self.plan_llm_memory(
                n_layers, hidden_dim, n_heads, seq_len,
                weight_dtype_bytes, kv_dtype_bytes, batch_size)
            if plan['fits']:
                return plan
        return self.plan_llm_memory(
            n_layers, hidden_dim, n_heads, 64,
            weight_dtype_bytes, kv_dtype_bytes, batch_size)

print("=== NPU内存规划 (7B模型) ===")

configs = [
    ('7B INT4 + KV INT8', 1, 1, 8192),
    ('7B INT4 + KV INT4', 1, 0.5, 8192),
    ('7B INT8 + KV INT8', 2, 1, 8192),
]

for name, w_bytes, kv_bytes, budget_mb in configs:
    planner = NPUMemoryPlanner(budget_mb)
    plan = planner.find_max_seq_len(
        n_layers=32, hidden_dim=4096, n_heads=32,
        weight_dtype_bytes=w_bytes, kv_dtype_bytes=kv_bytes, batch_size=1)
    print(f"\n--- {name} (预算: {budget_mb}MB) ---")
    print(f"  权重: {plan['weights_mb']:.0f}MB")
    print(f"  KV Cache: {plan['kv_cache_mb']:.0f}MB")
    print(f"  激活值: {plan['activations_mb']:.0f}MB")
    print(f"  总计: {plan['total_mb']:.0f}MB / {plan['budget_mb']:.0f}MB")
    print(f"  最大序列长度: {plan['max_seq_len']}")

print(f"\n=== 产业级内存规划要点 ===")
print(f"1. 权重占大头，量化是降低内存的最有效手段")
print(f"2. KV Cache随序列长度线性增长，长序列场景需KV量化")
print(f"3. 激活值通常较小（batch=1时），但需预留SRAM空间")
print(f"4. 实际部署需额外预留OS/框架开销（约500MB-1GB）")
print(f"5. 权重按需加载(weight streaming)可进一步降低峰值内存，但增加延迟")

---
## 5.1.7 NPU性能Profile与调试

产业级NPU部署必须经过严格的性能profile和精度验证，否则难以定位瓶颈和排查问题。

### Profile方法论

| Profile维度 | 工具 | 关键指标 |
|------------|------|----------|
| **算子级耗时** | QNN Profiler / msprof / Instruments | 各算子延迟、MAC利用率 |
| **内存占用** | 运行时统计 | 峰值内存、SRAM使用率、DRAM访问量 |
| **功耗** | 硬件功耗计 / 系统API | 平均功耗、峰值功耗、能耗/token |
| **精度验证** | 逐层对比工具 | 各层输出余弦相似度、端到端PPL变化 |
| **NPU利用率** | 硬件计数器 | MAC利用率、DMA利用率、流水线气泡 |

### 常见性能瓶颈与优化

| 瓶颈类型 | 症状 | 优化方向 |
|---------|------|----------|
| **CPU回退** | 个别算子耗时异常高 | 算子分解/替换，减少CPU↔NPU数据搬运 |
| **SRAM溢出** | 大量DRAM↔SRAM数据搬运 | Tiling策略优化，减少中间结果暂存 |
| **MAC利用率低** | decode阶段算力浪费 | 增大batch、算子融合减少流水线气泡 |
| **DMA瓶颈** | 数据搬运耗时占比高 | 双缓冲、权重预取、计算与搬运重叠 |
| **量化精度损失** | 输出质量明显下降 | 混合精度、敏感层保持高精度 |

### 精度验证流程

1. **逐层对比**：将NPU推理的每层输出与PyTorch参考实现对比，计算余弦相似度
2. **敏感层定位**：找到相似度最低的层，通常是量化的瓶颈层
3. **混合精度调整**：对敏感层提升精度，重新编译验证
4. **端到端验证**：在测试集上对比PPL和下游任务指标

In [ ]:
class NPUProfiler:
    """NPU性能Profile模拟器"""
    def __init__(self, npu_name: str):
        self.npu_name = npu_name

    def profile_llm_layer(self, layer_idx, hidden_dim, n_heads, seq_len,
                          is_decode=False, weight_dtype='int4', kv_dtype='int8'):
        """模拟单层Transformer的Profile数据"""
        head_dim = hidden_dim // n_heads
        dtype_bytes = {'int4': 0.5, 'int8': 1, 'fp16': 2}[weight_dtype]
        kv_bytes = {'int4': 0.5, 'int8': 1, 'fp16': 2}[kv_dtype]

        qkv_flops = 2 * seq_len * hidden_dim * hidden_dim * 3
        attn_flops = 2 * seq_len * seq_len * hidden_dim
        out_flops = 2 * seq_len * hidden_dim * hidden_dim
        ffn_flops = 2 * seq_len * hidden_dim * hidden_dim * 3
        total_flops = qkv_flops + attn_flops + out_flops + ffn_flops

        weight_bytes = hidden_dim * hidden_dim * (4 + 3) * dtype_bytes
        kv_bytes_total = 2 * seq_len * hidden_dim * kv_bytes
        activation_bytes = seq_len * hidden_dim * 2 * 2
        total_bytes = weight_bytes + kv_bytes_total + activation_bytes

        npu_peak_tops = 75
        dram_bw_gbs = 60
        compute_time_us = total_flops / (npu_peak_tops * 1e12 / 1e6) * 1000
        memory_time_us = total_bytes / (dram_bw_gbs * 1e9 / 1e6)
        actual_time_us = max(compute_time_us, memory_time_us)

        bottleneck = 'compute' if compute_time_us > memory_time_us else 'memory'
        mac_util = min(compute_time_us / actual_time_us, 1.0) if actual_time_us > 0 else 0

        return {
            'layer': layer_idx,
            'phase': 'decode' if is_decode else 'prefill',
            'total_flops_G': total_flops / 1e9,
            'total_bytes_MB': total_bytes / 1e6,
            'compute_time_us': compute_time_us,
            'memory_time_us': memory_time_us,
            'actual_time_us': actual_time_us,
            'bottleneck': bottleneck,
            'mac_utilization': mac_util,
        }

    def profile_full_model(self, n_layers, hidden_dim, n_heads, seq_len):
        """Profile完整模型"""
        prefill_layers = [self.profile_llm_layer(i, hidden_dim, n_heads, seq_len)
                          for i in range(n_layers)]
        decode_layers = [self.profile_llm_layer(i, hidden_dim, n_heads, 1, is_decode=True)
                         for i in range(n_layers)]

        prefill_total = sum(l['actual_time_us'] for l in prefill_layers)
        decode_total = sum(l['actual_time_us'] for l in decode_layers)

        return {
            'prefill_ms': prefill_total / 1000,
            'decode_ms_per_token': decode_total / 1000,
            'prefill_bottleneck': 'compute' if prefill_layers[0]['bottleneck'] == 'compute' else 'memory',
            'decode_bottleneck': 'memory' if decode_layers[0]['bottleneck'] == 'memory' else 'compute',
            'decode_mac_util': decode_layers[0]['mac_utilization'],
        }

profiler = NPUProfiler('qualcomm_hexagon')

print("=== 7B模型NPU Profile (高通Hexagon) ===")
result = profiler.profile_full_model(n_layers=32, hidden_dim=4096, n_heads=32, seq_len=512)
print(f"\nPrefill阶段:")
print(f"  总延迟: {result['prefill_ms']:.1f} ms")
print(f"  瓶颈: {result['prefill_bottleneck']}")
print(f"\nDecode阶段:")
print(f"  每token延迟: {result['decode_ms_per_token']:.2f} ms")
print(f"  瓶颈: {result['decode_bottleneck']}")
print(f"  MAC利用率: {result['decode_mac_util']:.2%}")

print(f"\n--- 单层详细Profile ---")
layer_profile = profiler.profile_llm_layer(0, 4096, 32, 512)
print(f"{'指标':<25} {'Prefill':<15}")
print("-" * 40)
print(f"{'计算量(GFLOPs)':<25} {layer_profile['total_flops_G']:<15.2f}")
print(f"{'数据搬运(MB)':<25} {layer_profile['total_bytes_MB']:<15.2f}")
print(f"{'计算时间(μs)':<25} {layer_profile['compute_time_us']:<15.1f}")
print(f"{'访存时间(μs)':<25} {layer_profile['memory_time_us']:<15.1f}")
print(f"{'实际时间(μs)':<25} {layer_profile['actual_time_us']:<15.1f}")
print(f"{'瓶颈':<25} {layer_profile['bottleneck']:<15}")

decode_profile = profiler.profile_llm_layer(0, 4096, 32, 1, is_decode=True)
print(f"\n{'指标':<25} {'Decode':<15}")
print("-" * 40)
print(f"{'计算量(GFLOPs)':<25} {decode_profile['total_flops_G']:<15.4f}")
print(f"{'数据搬运(MB)':<25} {decode_profile['total_bytes_MB']:<15.2f}")
print(f"{'实际时间(μs)':<25} {decode_profile['actual_time_us']:<15.1f}")
print(f"{'瓶颈':<25} {decode_profile['bottleneck']:<15}")
print(f"{'MAC利用率':<25} {decode_profile['mac_utilization']:<15.2%}")

print(f"\n=== Profile驱动的优化决策 ===")
print(f"1. Prefill是compute-bound → 优化方向: 算子融合、Flash Attention、增大batch")
print(f"2. Decode是memory-bound → 优化方向: 量化(W4减少4x搬运)、权重预取、双缓冲")
print(f"3. Decode MAC利用率极低 → 优化方向: 增大batch、投机解码、多token并行")
print(f"4. CPU回退算子 → 优化方向: 算子分解、自定义NPU算子注册")

---
## 5.1.8 异构调度与CPU+NPU协同

实际端侧部署中，LLM推理不可能100%在NPU上执行，需要CPU和NPU协同工作。

### 典型的CPU+NPU分工

| 组件 | 执行设备 | 原因 |
|------|---------|------|
| **Token Embedding** | CPU | 查表操作，NPU效率不高 |
| **QKV投影 + Attention计算** | NPU | 密集矩阵乘，NPU核心优势 |
| **FFN (SwiGLU)** | NPU | 密集矩阵乘，NPU核心优势 |
| **LayerNorm / RMSNorm** | NPU | 逐元素+归约，NPU支持 |
| **RoPE** | NPU | 逐元素乘加，NPU支持 |
| **Top-K / Sampling** | CPU | 动态排序，NPU不支持 |
| **KV Cache管理** | CPU | 动态索引操作，CPU更灵活 |
| **Tokenizer** | CPU | 字符串处理，非张量运算 |

### 异构调度策略

1. **子图级调度**：将计算图划分为NPU子图和CPU子图，子图间通过共享内存传递数据
2. **流水线并行**：NPU执行第i层时，CPU同步准备第i+1层的输入
3. **双缓冲**：NPU处理当前batch时，DMA预取下一个batch的权重到SRAM

### CPU↔NPU数据搬运优化

$$T_{\text{total}} = T_{\text{NPU}} + T_{\text{CPU}} + \sum_{i} T_{\text{copy},i}$$

目标：最小化$\sum_i T_{\text{copy},i}$，通过：
- 共享内存（零拷贝）
- 计算与搬运重叠（异步DMA）
- 减少切换次数（算子融合减少子图数量）

In [ ]:
class HeterogeneousScheduler:
    """CPU+NPU异构调度模拟器"""
    def __init__(self):
        self.npu_ops = {'qkv_proj', 'attn_score', 'attn_output', 'out_proj',
                        'gate_proj', 'up_proj', 'down_proj', 'silu',
                        'layer_norm', 'rope', 'softmax', 'elementwise'}
        self.cpu_ops = {'embedding', 'topk', 'sampling', 'kv_cache_update', 'tokenizer'}
        self.copy_latency_us = 15

    def schedule_layer(self, ops: List[str]) -> List[Dict]:
        """调度单层Transformer的算子"""
        schedule = []
        current_device = None
        current_group = []

        for op in ops:
            device = 'NPU' if op in self.npu_ops else 'CPU'
            if device != current_device and current_group:
                schedule.append({
                    'device': current_device,
                    'ops': current_group[:],
                    'copy_overhead_us': self.copy_latency_us if (schedule and current_device != schedule[-1]['device']) else 0
                })
                current_group = []
            current_device = device
            current_group.append(op)

        if current_group:
            schedule.append({'device': current_device, 'ops': current_group[:], 'copy_overhead_us': 0})

        n_copies = sum(1 for i in range(len(schedule)-1) if schedule[i]['device'] != schedule[i+1]['device'])
        total_copy_us = n_copies * self.copy_latency_us

        return schedule, n_copies, total_copy_us

scheduler = HeterogeneousScheduler()

layer_ops = ['embedding', 'layer_norm', 'rope', 'qkv_proj', 'attn_score',
             'softmax', 'attn_output', 'out_proj', 'layer_norm',
             'gate_proj', 'silu', 'up_proj', 'elementwise', 'down_proj',
             'kv_cache_update']

schedule, n_copies, copy_us = scheduler.schedule_layer(layer_ops)

print("=== 单层Transformer异构调度 ===")
for i, group in enumerate(schedule):
    print(f"  ① {group['device']}: {', '.join(group['ops'])}")
    if i < len(schedule) - 1 and schedule[i]['device'] != schedule[i+1]['device']:
        print(f"    ↓ (数据搬运: {scheduler.copy_latency_us}μs)")

print(f"\n设备切换次数: {n_copies}")
print(f"数据搬运总延迟: {copy_us}μs")

print(f"\n=== 优化策略对比 ===")
strategies = [
    ('基线(无优化)', n_copies, copy_us, '每次设备切换都搬运数据'),
    ('算子融合', max(1, n_copies - 2), max(0, (n_copies-2)) * scheduler.copy_latency_us,
     '融合NPU上的连续算子，减少子图数量'),
    ('共享内存+异步DMA', n_copies, copy_us // 3,
     '零拷贝+计算与搬运重叠，有效延迟降低3x'),
    ('全NPU化(理想)', 0, 0,
     '所有算子NPU化，无切换开销(需自定义NPU算子)'),
]
print(f"{'策略':<20} {'切换次数':<10} {'搬运延迟(μs)':<15} {'说明':<30}")
print("-" * 75)
for name, copies, latency, desc in strategies:
    print(f"{name:<20} {copies:<10} {latency:<15.0f} {desc:<30}")

print(f"\n=== 产业实践要点 ===")
print(f"1. 算子融合是最有效的减少设备切换的方法")
print(f"2. 共享内存(零拷贝)可消除CPU↔NPU间的数据拷贝")
print(f"3. 异步DMA允许NPU计算和权重预取并行执行")
print(f"4. 将TopK/Sampling等CPU算子用近似算法替代可进一步减少切换")
print(f"5. 实际部署中，CPU+NPU切换开销约占总延迟的5-15%")

---
## 5.1.9 实际部署案例与性能基准

### 端侧LLM部署性能基准（2025年数据）

| 模型 | 量化 | 平台 | 推理方式 | Prefill | Decode (tok/s) | 内存占用 |
|------|------|------|---------|---------|---------------|----------|
| Qwen2.5-1.5B | W4A16 | 骁龙8 Elite NPU | QNN | ~15ms/512tok | 25-35 | ~1.2GB |
| Qwen2.5-3B | W4A16 | 骁龙8 Elite NPU | QNN | ~30ms/512tok | 15-20 | ~2.2GB |
| Llama-3.1-8B | W4A16 | 骁龙8 Elite NPU | QNN | ~80ms/512tok | 8-12 | ~5GB |
| Phi-3-mini-3.8B | W4A16 | M4 MacBook ANE | Core ML | ~25ms/512tok | 20-30 | ~2.8GB |
| Qwen2.5-1.5B | W4A16 | M4 MacBook ANE | Core ML | ~12ms/512tok | 30-40 | ~1.2GB |
| Llama-3.1-8B | Q4_K_M | M4 Max CPU | llama.cpp | ~50ms/512tok | 15-25 | ~5.5GB |
| Qwen2.5-7B | W8A8 | 昇腾310P | CANN | ~40ms/512tok | 12-18 | ~8GB |

> 注：以上数据为典型值，实际性能因SDK版本、驱动、散热条件而异。

### 关键性能洞察

1. **1.5B-3B模型是端侧NPU的甜蜜点**：在4-8GB内存预算内，可达到15-35 tok/s的实时对话速度
2. **7B+模型需要INT4量化**：INT8的7B模型约7GB，超出大多数手机NPU的内存预算
3. **NPU vs CPU推理**：同模型同量化下，NPU比CPU快2-5x，功耗低3-10x
4. **ANE对FP16优化最佳**：苹果ANE的FP16推理效率高于INT8，与高通NPU相反
5. **实际部署的内存开销**：除模型权重外，还需预留KV Cache、运行时、OS开销

In [ ]:
benchmarks = [
    {'model': 'Qwen2.5-1.5B', 'quant': 'W4A16', 'platform': '骁龙8 Elite', 'backend': 'QNN NPU',
     'prefill_ms_512': 15, 'decode_tps': 30, 'memory_mb': 1200},
    {'model': 'Qwen2.5-3B', 'quant': 'W4A16', 'platform': '骁龙8 Elite', 'backend': 'QNN NPU',
     'prefill_ms_512': 30, 'decode_tps': 18, 'memory_mb': 2200},
    {'model': 'Llama-3.1-8B', 'quant': 'W4A16', 'platform': '骁龙8 Elite', 'backend': 'QNN NPU',
     'prefill_ms_512': 80, 'decode_tps': 10, 'memory_mb': 5000},
    {'model': 'Phi-3-mini-3.8B', 'quant': 'W4A16', 'platform': 'M4 MacBook', 'backend': 'Core ML ANE',
     'prefill_ms_512': 25, 'decode_tps': 25, 'memory_mb': 2800},
    {'model': 'Qwen2.5-1.5B', 'quant': 'W4A16', 'platform': 'M4 MacBook', 'backend': 'Core ML ANE',
     'prefill_ms_512': 12, 'decode_tps': 35, 'memory_mb': 1200},
    {'model': 'Llama-3.1-8B', 'quant': 'Q4_K_M', 'platform': 'M4 Max', 'backend': 'llama.cpp CPU',
     'prefill_ms_512': 50, 'decode_tps': 20, 'memory_mb': 5500},
]

print("=== 端侧LLM部署性能基准 ===")
print(f"{'模型':<20} {'量化':<8} {'平台':<15} {'后端':<15} {'Prefill(ms)':<12} {'Decode(tok/s)':<14} {'内存(MB)':<10}")
print("-" * 94)
for b in benchmarks:
    print(f"{b['model']:<20} {b['quant']:<8} {b['platform']:<15} {b['backend']:<15} "
          f"{b['prefill_ms_512']:<12} {b['decode_tps']:<14} {b['memory_mb']:<10}")

print(f"\n=== 端侧部署选型建议 ===")
print(f"\n1. 模型选择:")
print(f"   - 手机端(6-8GB RAM): 1.5B-3B模型 + W4A16量化")
print(f"   - 平板/笔记本(8-16GB): 3B-7B模型 + W4A16量化")
print(f"   - 车载/边缘服务器(16GB+): 7B-13B模型 + W4A16或W8A8")
print(f"\n2. NPU选择:")
print(f"   - 高通Hexagon: INT4支持最好，Android端首选")
print(f"   - 苹果ANE: FP16效率最高，iOS/macOS首选")
print(f"   - 华为昇腾: 算力最高，国产化场景首选")
print(f"\n3. 部署框架:")
print(f"   - 高通: QNN SDK → Context Binary")
print(f"   - 苹果: coremltools → Core ML")
print(f"   - 华为: AMCT + ATC → OM模型")
print(f"   - 通用CPU: llama.cpp → GGUF")
print(f"\n4. 关键限制:")
print(f"   - 内存是第一约束: 模型+KV+运行时必须fit进设备内存")
print(f"   - 散热是第二约束: 持续推理会触发降频，需热管理策略")
print(f"   - 算子兼容性是第三约束: 不支持的算子必须分解或回退")

---
## 5.1.10 NPU自定义算子开发与注册

当NPU原生不支持的算子频繁回退CPU时，回退开销会严重影响端到端性能。此时需要开发NPU自定义算子，将关键算子直接映射到NPU硬件指令。

### 为什么需要自定义算子？

CPU回退的代价不仅是算子本身的执行延迟，还包括：
- **数据搬运开销**：NPU→CPU→NPU的两次数据拷贝，约$10-50\mu s$
- **流水线中断**：CPU回退打断NPU流水线，后续算子需等待
- **内存碎片**：频繁的CPU↔NPU数据交换导致内存管理复杂化

对于LLM推理，以下算子最值得自定义实现：

| 算子 | CPU回退代价 | 自定义收益 | 实现难度 |
|------|-----------|----------|----------|
| **RoPE** | 每层2次回退，32层=64次 | 高（消除64次数据搬运） | 低（逐元素运算） |
| **SwiGLU** | SiLU部分NPU不支持 | 中 | 低（SiLU≈Sigmoid×x） |
| **RMSNorm** | pow/sqrt部分回退 | 中 | 中（需定点开方近似） |
| **Top-K** | 每decode步1次回退 | 中（仅1次但延迟高） | 高（需排序算法） |
| **KV Cache更新** | Scatter/Gather回退 | 高（每层每步） | 中（需自定义索引） |

### 各厂商自定义算子注册机制

| 厂商 | 注册机制 | 开发语言 | 编译方式 |
|------|---------|---------|----------|
| **高通QNN** | QNN OpDef注册 + 自定义OpPackage | C/C++ | 编译为.so动态库，运行时加载 |
| **华为CANN** | Ascend C算子开发 + OPP注册 | Ascend C (类C++) | 编译为.o，注册到算子包(OPP) |
| **苹果Core ML** | MLProgram自定义Compute Unit | Metal/Swift | Xcode编译到App Bundle |

### QNN自定义算子开发流程

```
① 定义算子接口 (OpDef)
   → 指定算子名称、输入/输出张量规格、支持的精度
② 实现算子逻辑 (execute函数)
   → 在execute函数中实现算子计算逻辑
   → 可调用HVX/HTA指令进行硬件加速
③ 编译为OpPackage (.so)
   → qnn-op-package-generator --op_package_config my_ops.json
   → 生成libCustomOpPackage.so
④ 集成到模型编译
   → qnn-onnx-converter --op_packages libCustomOpPackage.so ...
```

### CANN自定义算子开发流程

```
① 使用Ascend C编写算子
   → 继承Kernel类，实现Init()和Process()方法
   → 使用TIK(DSL)或TBE(Tensor Boost Engine) API
② 编译算子到OPP
   → msopgen register --op_type RoPE --so_path ./build/libRoPE.so
   → 注册到算子包路径 $ASCEND_HOME/opp/vendors/customize/
③ ATC编译时自动发现自定义算子
   → atc --model=model.onnx --output=model.om --soc_version=Ascend310P3
```

### 自定义算子的性能决策模型

是否值得开发自定义NPU算子，取决于收益-成本分析：

$$\text{ROI} = \frac{N_{\text{calls}} \times (T_{\text{fallback}} - T_{\text{custom}})}{T_{\text{dev}} + T_{\text{maintain}}}$$

其中：
- $N_{\text{calls}}$：单次推理中算子调用次数
- $T_{\text{fallback}}$：CPU回退延迟（含数据搬运）
- $T_{\text{custom}}$：自定义NPU算子延迟
- $T_{\text{dev}}$：开发时间成本
- $T_{\text{maintain}}$：维护时间成本（SDK升级、跨平台适配）

**经验法则**：
- 单次推理调用次数 > 10（如RoPE每层调用，32层=64次）→ 值得自定义
- 单次推理调用次数 < 3（如Top-K仅1次）→ CPU回退更经济
- 部署规模 > 10万设备 → 维护成本可摊薄，值得投入

### 产业实践：RoPE算子的NPU自定义实现

RoPE（Rotary Position Embedding）是LLM中每层都调用的算子，其数学定义为：

$$q_{\text{rot}} = q \cos\theta + q_{\text{shift}} \sin\theta$$

其中$\theta = \text{freq} \times \text{pos}$，$q_{\text{shift}}$是$q$的奇偶位互换。

NPU自定义实现的关键优化：
1. **预计算cos/sin表**：将cos/sin值预计算为常量张量，推理时仅需查表
2. **融合到QKV投影**：将RoPE融合到QKV MatMul之后，避免中间结果写回DRAM
3. **INT8查表**：将cos/sin表量化为INT8，与INT8的Q/K直接相乘
4. **向量化实现**：利用NPU向量处理器的SIMD指令并行计算

| 实现方式 | 延迟(μs/层) | 数据搬运 | 总延迟(32层) |
|---------|------------|---------|------------|
| CPU回退 | 25 | 2次NPU↔CPU | 800μs |
| 分解为逐元素运算 | 8 | 0 | 256μs |
| 自定义NPU算子 | 3 | 0 | 96μs |

In [ ]:
class NPUCustomOpRegistry:
    """NPU自定义算子注册与调度决策模拟器"""
    def __init__(self):
        self.registry = {}
        self.implementations = {}

    def register_custom_op(self, op_name, latency_us, dev_cost_days, maintain_cost_days):
        """注册自定义NPU算子"""
        self.registry[op_name] = {
            'mode': 'custom',
            'latency_us': latency_us,
            'dev_cost_days': dev_cost_days,
            'maintain_cost_days': maintain_cost_days,
        }

    def register_decomposed_op(self, op_name, sub_ops, latency_us):
        """注册分解后的算子组合"""
        self.registry[op_name] = {
            'mode': 'decomposed',
            'sub_ops': sub_ops,
            'latency_us': latency_us,
        }

    def register_cpu_fallback(self, op_name, latency_us, copy_overhead_us=20):
        """注册CPU回退算子"""
        self.registry[op_name] = {
            'mode': 'cpu_fallback',
            'latency_us': latency_us,
            'copy_overhead_us': copy_overhead_us,
        }

    def should_customize(self, op_name, calls_per_inference, deploy_scale=1):
        """判断是否值得为该算子开发自定义NPU实现"""
        info = self.registry.get(op_name)
        if not info:
            return False, '算子未注册'

        if info['mode'] == 'custom':
            return True, '已有自定义实现'

        cpu_total_us = (info['latency_us'] + info.get('copy_overhead_us', 0)) * calls_per_inference

        if info['mode'] == 'decomposed':
            decomposed_total_us = info['latency_us'] * calls_per_inference
            saving_per_inference = cpu_total_us - decomposed_total_us
            if saving_per_inference < 50:
                return False, f'分解方案已足够高效(节省{saving_per_inference:.0f}μs/推理)'

        if calls_per_inference >= 10:
            return True, f'调用频率高({calls_per_inference}次/推理)，累计节省显著'
        elif calls_per_inference >= 3 and deploy_scale >= 100000:
            return True, f'部署规模大({deploy_scale}台)，维护成本可摊薄'
        else:
            return False, f'调用频率低({calls_per_inference}次/推理)，CPU回退更经济'

    def generate_scheduling_plan(self, n_layers=32):
        """生成完整的算子调度方案"""
        plan = []
        for op_name, info in self.registry.items():
            calls = n_layers if op_name in ['rope', 'rmsnorm', 'kv_cache_update'] else 1
            if op_name == 'swiglu':
                calls = n_layers
            should, reason = self.should_customize(op_name, calls)
            plan.append({
                'op': op_name,
                'mode': info['mode'],
                'calls_per_inference': calls,
                'should_customize': should,
                'reason': reason,
            })
        return plan

registry = NPUCustomOpRegistry()

registry.register_cpu_fallback('rope', latency_us=25, copy_overhead_us=20)
registry.register_decomposed_op('rope', sub_ops=['gather', 'mul', 'add'], latency_us=8)
registry.register_custom_op('rope_custom', latency_us=3, dev_cost_days=3, maintain_cost_days=1)

registry.register_cpu_fallback('rmsnorm', latency_us=15, copy_overhead_us=20)
registry.register_decomposed_op('rmsnorm', sub_ops=['pow', 'sum', 'sqrt', 'div', 'mul'], latency_us=6)

registry.register_cpu_fallback('swiglu', latency_us=10, copy_overhead_us=20)
registry.register_decomposed_op('swiglu', sub_ops=['matmul', 'silu', 'matmul', 'mul', 'matmul'], latency_us=5)

registry.register_cpu_fallback('topk', latency_us=50, copy_overhead_us=15)

registry.register_cpu_fallback('kv_cache_update', latency_us=8, copy_overhead_us=20)
registry.register_custom_op('kv_cache_update_custom', latency_us=2, dev_cost_days=5, maintain_cost_days=2)

print("=== NPU自定义算子调度决策 ===")
print(f"{'算子':<25} {'当前模式':<15} {'调用次数/推理':<15} {'是否自定义':<10} {'决策原因'}")
print("-" * 100)

plan = registry.generate_scheduling_plan(n_layers=32)
for item in plan:
    should_str = '✓ 是' if item['should_customize'] else '✗ 否'
    print(f"{item['op']:<25} {item['mode']:<15} {item['calls_per_inference']:<15} {should_str:<10} {item['reason']}")

print(f"\n=== 自定义算子开发优先级 ===")
priority_ops = [p for p in plan if p['should_customize']]
priority_ops.sort(key=lambda x: -x['calls_per_inference'])
for i, op in enumerate(priority_ops, 1):
    print(f"  优先级{i}: {op['op']} (调用{op['calls_per_inference']}次/推理)")

print(f"\n=== 产业实践要点 ===")
print(f"1. RoPE是最高优先级自定义算子：32层×2=64次调用，累计节省约700μs")
print(f"2. KV Cache更新次之：32层×1=32次调用，自定义后可消除32次CPU回退")
print(f"3. Top-K仅1次调用，CPU回退更经济，但可用近似Top-K算法加速")
print(f"4. 自定义算子的维护成本常被低估：SDK升级、跨芯片适配、精度回归测试")
print(f"5. 建议先用分解方案上线，Profile确认瓶颈后再投入自定义开发")

---
## 5.1.11 NPU编译器内幕与计算图优化

理解NPU编译器的工作原理是产业级部署工程师的核心竞争力。编译器决定了模型在NPU上的实际执行效率，同样的模型和硬件，不同的编译策略可能导致3-10x的性能差异。

### NPU编译器的三阶段架构

```
前端 (Front-end)           中端 (Middle-end)           后端 (Back-end)
┌─────────────────┐     ┌──────────────────┐     ┌──────────────────┐
│ ONNX/TorchScript│     │ 计算图优化        │     │ 指令生成与调度    │
│ → 内部IR        │ ──→ │ 算子融合          │ ──→ │ Tiling/分块      │
│ 算子语义解析     │     │ 常量折叠          │     │ 内存规划          │
│ 类型推导        │     │ 死代码消除        │     │ 指令调度          │
│ Shape推导       │     │ 布局变换          │     │ 寄存器分配        │
└─────────────────┘     └──────────────────┘     └──────────────────┘
```

### 关键编译Pass详解

| 编译Pass | 功能 | 对性能的影响 | 产业实践要点 |
|---------|------|------------|-------------|
| **算子融合** | 将多个算子合并为一个，减少中间结果写回DRAM | 高：减少30-50% DRAM访问 | 融合QKV+RoPE、SiLU+Mul、LN+Residual是LLM关键融合 |
| **常量折叠** | 编译期计算常量表达式（如RoPE的cos/sin表） | 中：减少运行时计算 | RoPE预计算表是典型应用，但增加模型体积 |
| **布局变换** | 将张量排布从NCHW转为NPU最优的NHWC/自定义布局 | 高：影响MAC利用率 | 不同NPU最优布局不同，错误布局导致2-3x性能下降 |
| **Tiling/分块** | 将大算子切分为适配SRAM的小块 | 关键：决定SRAM溢出与否 | tile_size选择需平衡SRAM使用和循环开销 |
| **内存规划** | 分析张量生命周期，复用不再需要的内存 | 高：减少30-50%峰值内存 | LLM中KV Cache与激活值可分时复用 |
| **指令调度** | 安排算子执行顺序，最大化流水线并行 | 中：减少10-20%延迟 | 双缓冲+权重预取是关键调度策略 |

### 内存规划：张量生命周期分析

NPU编译器通过分析张量的生命周期来规划内存分配，核心思想是**互不重叠的生命周期可以共享同一块内存**：

```
时间轴:  t0    t1    t2    t3    t4    t5    t6
QKV输出: ████████████                          (t0-t1, 可释放)
Attn分数:      ████████████                     (t1-t2, 复用QKV内存)
Attn输出:           ████████████                (t2-t3, 复用Attn分数内存)
FFN中间:                 ████████████           (t3-t4, 复用Attn输出内存)
FFN输出:                      ████████████      (t4-t5, 复用FFN中间内存)
```

通过生命周期分析，单层Transformer的峰值内存从$\sum \text{all tensors}$降至$max(\text{largest tensor})$。

### 各厂商编译器对比

| 特性 | QNN Graph Optimizer | Core ML Compiler | CANN ACE | TVM (通用) |
|------|-------------------|-----------------|----------|------------|
| **融合策略** | 基于模式的规则融合 | ANE专用融合规则 | 达芬奇架构感知融合 | AutoTVM搜索最优融合 |
| **Tiling** | 基于SRAM容量自动分块 | 固定分块策略 | Cube维度感知分块 | 基于搜索的分块 |
| **内存规划** | 线性扫描+贪心 | 静态内存池 | 多级内存规划 | Graph-level内存优化 |
| **调度策略** | HTP流水线调度 | ANE数据流调度 | 达芬奇三核协同 | Auto-scheduling |
| **可调试性** | QNN Profiler+Debug | Instruments | msprof+对比工具 | TVM Debug Runtime |
| **编译时间** | 5-30分钟(7B) | 2-10分钟 | 10-60分钟 | 30-120分钟(AutoTVM) |

In [ ]:
class NPUCompilerSimulator:
    """NPU编译器关键Pass模拟器"""
    def __init__(self, sram_capacity_kb=4096, dtype_bytes=2):
        self.sram_capacity = sram_capacity_kb * 1024
        self.dtype_bytes = dtype_bytes

    def analyze_fusion_opportunities(self, hidden_dim, n_heads, seq_len):
        """分析LLM单层的算子融合机会"""
        head_dim = hidden_dim // n_heads
        b = self.dtype_bytes

        no_fusion_tensors = []
        no_fusion_tensors.append(('QKV输出', 3 * seq_len * hidden_dim * b))
        no_fusion_tensors.append(('RoPE中间', 2 * seq_len * hidden_dim * b))
        no_fusion_tensors.append(('Attn分数', n_heads * seq_len * seq_len * b))
        no_fusion_tensors.append(('Attn输出', seq_len * hidden_dim * b))
        no_fusion_tensors.append(('Residual', seq_len * hidden_dim * b))
        no_fusion_tensors.append(('FFN gate', seq_len * hidden_dim * b))
        no_fusion_tensors.append(('SiLU输出', seq_len * hidden_dim * b))
        no_fusion_tensors.append(('FFN up', seq_len * hidden_dim * b))
        no_fusion_tensors.append(('FFN输出', seq_len * hidden_dim * b))
        no_fusion_total = sum(s for _, s in no_fusion_tensors)

        fused_tensors = []
        fused_tensors.append(('QKV+RoPE融合', seq_len * hidden_dim * b * 3))
        fused_tensors.append(('Attn融合(QK+Softmax+AV)', n_heads * seq_len * seq_len * b))
        fused_tensors.append(('Attn+O_proj+Residual融合', seq_len * hidden_dim * b))
        fused_tensors.append(('FFN融合(gate+SiLU+up+down)', seq_len * hidden_dim * b))
        fused_total = sum(s for _, s in fused_tensors)

        return {
            'no_fusion': {'tensors': no_fusion_tensors, 'total_bytes': no_fusion_total},
            'fused': {'tensors': fused_tensors, 'total_bytes': fused_total},
            'sram_savings': no_fusion_total - fused_total,
            'sram_savings_pct': (no_fusion_total - fused_total) / no_fusion_total * 100,
        }

    def analyze_memory_reuse(self, hidden_dim, n_heads, seq_len):
        """分析张量生命周期与内存复用机会"""
        b = self.dtype_bytes
        tensor_sizes = {
            'QKV输出': 3 * seq_len * hidden_dim * b,
            'Attn分数': n_heads * seq_len * seq_len * b,
            'Attn输出': seq_len * hidden_dim * b,
            'FFN中间1': seq_len * hidden_dim * b,
            'FFN中间2': seq_len * hidden_dim * b,
            'FFN输出': seq_len * hidden_dim * b,
        }
        lifetimes = {
            'QKV输出': (0, 1),
            'Attn分数': (1, 2),
            'Attn输出': (2, 3),
            'FFN中间1': (3, 4),
            'FFN中间2': (3, 4),
            'FFN输出': (4, 5),
        }
        no_reuse = sum(tensor_sizes.values())
        max_concurrent = 0
        for t in range(6):
            concurrent = sum(tensor_sizes[name] for name, (start, end) in lifetimes.items() if start <= t < end)
            max_concurrent = max(max_concurrent, concurrent)
        return {
            'naive_total_mb': no_reuse / 1024 / 1024,
            'reuse_peak_mb': max_concurrent / 1024 / 1024,
            'savings_pct': (1 - max_concurrent / no_reuse) * 100,
        }

compiler = NPUCompilerSimulator(sram_capacity_kb=4096, dtype_bytes=2)

print("=== NPU编译器算子融合分析 (7B模型, seq_len=512) ===")
fusion = compiler.analyze_fusion_opportunities(hidden_dim=4096, n_heads=32, seq_len=512)
print(f"\n无融合 - 中间张量:")
for name, size in fusion['no_fusion']['tensors']:
    print(f"  {name}: {size/1024/1024:.2f} MB")
print(f"  总计: {fusion['no_fusion']['total_bytes']/1024/1024:.2f} MB")

print(f"\n融合后 - 中间张量:")
for name, size in fusion['fused']['tensors']:
    print(f"  {name}: {size/1024/1024:.2f} MB")
print(f"  总计: {fusion['fused']['total_bytes']/1024/1024:.2f} MB")
print(f"\nSRAM节省: {fusion['sram_savings']/1024/1024:.2f} MB ({fusion['sram_savings_pct']:.1f}%)")

print(f"\n=== 张量生命周期与内存复用分析 ===")
reuse = compiler.analyze_memory_reuse(hidden_dim=4096, n_heads=32, seq_len=512)
print(f"无复用峰值内存: {reuse['naive_total_mb']:.2f} MB")
print(f"复用后峰值内存: {reuse['reuse_peak_mb']:.2f} MB")
print(f"节省比例: {reuse['savings_pct']:.1f}%")

print(f"\n=== 产业级编译优化要点 ===")
print(f"1. 算子融合是减少DRAM访问的最有效手段，QKV+RoPE融合可消除2次DRAM写回")
print(f"2. 张量生命周期分析使峰值内存降低50%+，是7B模型fit进8GB设备的关键")
print(f"3. 编译时间与模型大小成正比，7B模型编译需10-60分钟")
print(f"4. AutoTVM等搜索式编译器可找到更优解，但编译时间增加10-100x")
print(f"5. 生产环境建议：开发期用快速编译迭代，发布前用搜索式编译优化")

---
## 5.1.12 跨平台NPU抽象层

产业级部署通常需要同时支持多种NPU硬件，直接使用各厂商SDK会导致代码碎片化。跨平台NPU抽象层通过统一的API屏蔽硬件差异，是大规模生产部署的必备基础设施。

### 为什么需要跨平台抽象层？

| 痛点 | 直接使用厂商SDK | 使用抽象层 |
|------|----------------|----------|
| **多平台适配** | 每个平台一套代码，维护成本O(N) | 一套代码多平台运行，维护成本O(1) |
| **SDK升级** | 各厂商SDK API不兼容，逐个迁移 | 抽象层处理兼容性，业务代码不变 |
| **算子兼容性** | 每个NPU手动检查和分解 | 抽象层自动处理回退和分解 |
| **性能调优** | 每个平台独立调优 | 抽象层提供跨平台调优接口 |
| **测试覆盖** | N×M个组合需要测试 | 统一测试框架 |

### 主流跨平台NPU抽象层对比

| 抽象层 | 支持的NPU | LLM支持 | 量化支持 | 生产就绪度 |
|--------|----------|---------|---------|----------|
| **ONNX Runtime** | QNN/NNAPI/CoreML/OpenVINO/DML | 有限(需自定义) | INT8/INT4(部分EP) | ★★★★★ |
| **TFLite** | NNAPI/CoreML/GPU/DSP | 不支持LLM | INT8/INT4 | ★★★★★ |
| **Apache TVM** | 任意NPU(需编译target) | MLC-LLM支持 | INT8/INT4/FP16 | ★★★★☆ |
| **ExecuTorch** | QNN/CoreML/XNNPACK | 原生LLM支持 | INT8/INT4 | ★★★☆☆ |
| **llama.cpp** | CPU only(通用) | 原生LLM支持 | Q2-Q8/GGUF | ★★★★★ |

### ONNX Runtime Execution Provider架构

```
┌──────────────────────────────────────────────────────┐
│                  应用层 (Python/C++/C#/Java)          │
├──────────────────────────────────────────────────────┤
│              ONNX Runtime 统一API                     │
│    InferenceSession → run() → 结果                    │
├──────┬──────┬───────┬────────┬────────┬──────────────┤
│ QNN  │NNAPI │CoreML │OpenVINO│  DML   │  CPU Fallback│
│  EP  │  EP  │  EP   │   EP   │  EP    │              │
├──────┴──────┴───────┴────────┴────────┴──────────────┤
│ Hexagon │ Android│ Apple │ Intel  │DirectX│  通用CPU   │
│  NPU    │  NPU   │  ANE  │  NPU   │ GPU   │           │
└─────────┴────────┴───────┴────────┴──────┴───────────┘
```

### 抽象层 vs 原生SDK的权衡

| 维度 | 原生SDK | 抽象层 |
|------|---------|--------|
| **峰值性能** | 最优（可利用全部硬件特性） | 次优（受限于抽象层通用接口） |
| **开发效率** | 低（每平台独立开发） | 高（一次开发多平台部署） |
| **算子覆盖** | 最全（厂商持续更新） | 受限（需等抽象层适配新算子） |
| **调试能力** | 强（厂商专业工具链） | 弱（抽象层屏蔽底层细节） |
| **版本兼容** | 差（API频繁变更） | 好（抽象层处理兼容性） |
| **适用场景** | 追求极致性能的单平台部署 | 多平台快速迭代和部署 |

### 产业实践建议

1. **MVP阶段**：使用llama.cpp(通用CPU)或ONNX Runtime快速验证可行性
2. **优化阶段**：在目标平台上使用原生SDK获取最优性能
3. **规模化阶段**：构建内部抽象层，统一多平台部署管线
4. **关键原则**：抽象层不应阻止你使用平台特定优化，应提供escape hatch

In [ ]:
@dataclass
class NPUBackend:
    name: str
    vendor: str
    abstraction_layer: str
    supports_llm: bool
    quant_support: List[str]
    latency_overhead_pct: float
    dev_effort_days: int

class CrossPlatformDeployManager:
    """跨平台NPU部署决策管理器"""
    BACKENDS = [
        NPUBackend('QNN NPU', 'Qualcomm', 'ONNX Runtime QNN EP', True,
                   ['int8', 'int4'], 0.0, 15),
        NPUBackend('Core ML ANE', 'Apple', 'ONNX Runtime CoreML EP', True,
                   ['fp16', 'int8'], 5.0, 10),
        NPUBackend('OpenVINO NPU', 'Intel', 'ONNX Runtime OpenVINO EP', False,
                   ['int8', 'fp16'], 8.0, 12),
        NPUBackend('NNAPI', 'Android', 'ONNX Runtime NNAPI EP', False,
                   ['int8', 'fp16'], 15.0, 5),
        NPUBackend('CPU (llama.cpp)', '通用', 'llama.cpp', True,
                   ['q2', 'q3', 'q4', 'q5', 'q6', 'q8'], 0.0, 2),
        NPUBackend('CPU (XNNPACK)', '通用', 'ExecuTorch XNNPACK', True,
                   ['int8'], 20.0, 8),
    ]

    def recommend_pipeline(self, target_platforms, model_size_b, max_dev_days=30):
        """根据目标平台和资源约束推荐部署管线"""
        platform_map = {
            'android_high': ['QNN NPU', 'NNAPI', 'CPU (llama.cpp)'],
            'android_mid': ['NNAPI', 'CPU (llama.cpp)'],
            'ios': ['Core ML ANE', 'CPU (llama.cpp)'],
            'macos': ['Core ML ANE', 'CPU (llama.cpp)'],
            'intel_edge': ['OpenVINO NPU', 'CPU (llama.cpp)'],
            'generic': ['CPU (llama.cpp)'],
        }
        backend_lookup = {b.name: b for b in self.BACKENDS}
        recommendations = {}
        total_dev_days = 0
        for platform in target_platforms:
            candidates = platform_map.get(platform, ['CPU (llama.cpp)'])
            for cand_name in candidates:
                backend = backend_lookup[cand_name]
                if backend.supports_llm or 'llama.cpp' in cand_name:
                    recommendations[platform] = backend
                    total_dev_days += backend.dev_effort_days
                    break
        return recommendations, total_dev_days

manager = CrossPlatformDeployManager()

scenarios = [
    ('手机端全平台', ['android_high', 'ios']),
    ('Android高中低端', ['android_high', 'android_mid']),
    ('全平台覆盖', ['android_high', 'android_mid', 'ios', 'intel_edge']),
]

print("=== 跨平台NPU部署管线推荐 ===")
for name, platforms in scenarios:
    recs, dev_days = manager.recommend_pipeline(platforms, model_size_b=3)
    print(f"\n--- {name} ---")
    for platform, backend in recs.items():
        print(f"  {platform}: {backend.name} (via {backend.abstraction_layer})")
        print(f"    量化支持: {backend.quant_support}, 延迟开销: +{backend.latency_overhead_pct}%")
    print(f"  总开发工时: ~{dev_days}天")

print(f"\n=== 跨平台部署策略决策树 ===")
print(f"1. 仅需CPU推理 → llama.cpp (最成熟, GGUF格式, 开箱即用)")
print(f"2. 仅需单平台NPU → 原生SDK (QNN/CoreML/CANN, 最优性能)")
print(f"3. 需要多平台NPU → ONNX Runtime + 多EP (统一API, 中等性能)")
print(f"4. 需要极致跨平台 → TVM/MLC-LLM (编译到任意target, 编译时间长)")
print(f"5. 快速原型验证 → ExecuTorch (Meta生态, LLM支持好, 生态发展中)")
print(f"\n关键原则: 抽象层的价值在于降低维护成本，而非提升单平台性能")
print(f"当部署规模>10万设备时，跨平台抽象层的ROI显著为正")

---
## 5.1.13 生产环境错误处理与可靠性保障

生产环境的NPU部署与实验室环境截然不同：硬件碎片化、驱动版本不一致、散热条件不可控、用户行为不可预测。必须建立完善的错误处理和降级机制，确保服务可用性。

### 生产环境常见故障模式

| 故障类型 | 发生频率 | 影响 | 检测方法 |
|---------|---------|------|----------|
| **NPU初始化失败** | 1-5% (驱动/固件问题) | 推理不可用 | 初始化超时/返回错误码 |
| **NPU运行时崩溃** | 0.1-1% (驱动bug/过热) | 当前推理失败 | 运行时异常/看门狗超时 |
| **内存不足** | 5-10% (长序列/多应用) | 推理失败或被系统杀死 | 内存监控+预检 |
| **精度异常** | 0.5-2% (量化/编译bug) | 输出质量下降 | 输出范围检查/PPL监控 |
| **热节流** | 持续推理时100% | 性能下降50-70% | 温度传感器+延迟监控 |
| **SDK版本不兼容** | 版本升级时10-20% | 编译/运行失败 | 版本检查+兼容性矩阵 |

### 三层降级策略

```
┌─────────────────────────────────────────────────┐
│  第一层: NPU推理 (最优性能)                       │
│  → INT4/INT8量化, NPU加速, 最低延迟              │
├─────────────────────────────────────────────────┤
│  第二层: CPU推理 (保底可用)                       │
│  → FP16/量化, CPU执行, 延迟增加3-5x              │
├─────────────────────────────────────────────────┤
│  第三层: 云端推理 (最终保障)                      │
│  → 请求转发到云端, 网络延迟+排队延迟              │
└─────────────────────────────────────────────────┘
```

### NPU能力检测与自适应

应用启动时必须检测NPU能力，根据检测结果选择最优推理路径：

1. **硬件检测**：NPU型号、驱动版本、可用内存
2. **能力检测**：支持的算子、精度、最大shape
3. **性能基准**：运行micro-benchmark测量实际算力
4. **决策**：根据检测结果选择推理配置

### OTA模型更新策略

NPU编译后的二进制与SDK版本强绑定，OTA更新需注意：
- **A/B分区**：保留旧版本模型，新版本验证通过后再切换
- **兼容性检查**：新模型二进制必须与当前SDK版本兼容
- **回滚机制**：新版本异常时快速回滚到旧版本
- **灰度发布**：先在1%设备上验证，逐步扩大到100%

In [ ]:
class ProductionReliabilityManager:
    """生产环境NPU可靠性管理器"""
    def __init__(self):
        self.degradation_levels = [
            {'level': 0, 'name': 'NPU INT4', 'expected_tps': 30, 'expected_latency_ms': 33},
            {'level': 1, 'name': 'NPU INT8', 'expected_tps': 18, 'expected_latency_ms': 56},
            {'level': 2, 'name': 'CPU INT4', 'expected_tps': 8, 'expected_latency_ms': 125},
            {'level': 3, 'name': 'CPU FP16', 'expected_tps': 4, 'expected_latency_ms': 250},
            {'level': 4, 'name': '云端推理', 'expected_tps': 0, 'expected_latency_ms': 500},
        ]
        self.health_status = {'npu_available': True, 'temperature_ok': True, 'memory_ok': True}

    def check_npu_health(self, temperature_c=45, available_memory_mb=4000, npu_driver_ok=True):
        """检查NPU健康状态"""
        self.health_status = {
            'npu_available': npu_driver_ok,
            'temperature_ok': temperature_c < 85,
            'memory_ok': available_memory_mb >= 2000,
        }
        return self.health_status

    def select_degradation_level(self):
        """根据健康状态选择降级级别"""
        if not self.health_status['npu_available']:
            return self.degradation_levels[2]
        if not self.health_status['temperature_ok']:
            return self.degradation_levels[1]
        if not self.health_status['memory_ok']:
            return self.degradation_levels[1]
        return self.degradation_levels[0]

    def simulate_ota_update(self, current_version, new_version, compatibility_matrix):
        """模拟OTA更新流程"""
        compatible = compatibility_matrix.get((current_version, new_version), False)
        return {
            'current': current_version,
            'target': new_version,
            'compatible': compatible,
            'strategy': 'ab_partition' if compatible else 'skip',
            'rollback_needed': not compatible,
        }

manager = ProductionReliabilityManager()

print("=== 生产环境降级策略模拟 ===")
scenarios = [
    ('正常状态', 45, 4000, True),
    ('高温降频', 90, 4000, True),
    ('内存不足', 45, 1500, True),
    ('NPU驱动异常', 45, 4000, False),
    ('多重故障', 90, 1500, False),
]

print(f"{'场景':<15} {'温度(°C)':<10} {'内存(MB)':<10} {'NPU':<8} {'选择策略':<15} {'预期tok/s':<10} {'预期延迟(ms)':<12}")
print("-" * 80)
for name, temp, mem, npu_ok in scenarios:
    manager.check_npu_health(temp, mem, npu_ok)
    level = manager.select_degradation_level()
    print(f"{name:<15} {temp:<10} {mem:<10} {'OK' if npu_ok else 'FAIL':<8} "
          f"{level['name']:<15} {level['expected_tps']:<10} {level['expected_latency_ms']:<12}")

print(f"\n=== OTA模型更新兼容性检查 ===")
compat_matrix = {
    ('qnn_2.20', 'qnn_2.22'): True,
    ('qnn_2.18', 'qnn_2.22'): False,
    ('cann_7.0', 'cann_7.2'): True,
    ('cann_6.3', 'cann_7.2'): False,
    ('coreml_7', 'coreml_8'): True,
}
for (cur, new), compat in compat_matrix.items():
    result = manager.simulate_ota_update(cur, new, compat_matrix)
    status = '✓ 兼容' if result['compatible'] else '✗ 不兼容(跳过/回滚)'
    print(f"  {cur} → {new}: {status}")

print(f"\n=== 生产环境可靠性保障要点 ===")
print(f"1. 永远不要假设NPU一定可用，必须有CPU降级方案")
print(f"2. 应用启动时检测NPU能力，运行时持续监控健康状态")
print(f"3. OTA更新必须A/B分区+兼容性检查+灰度发布+快速回滚")
print(f"4. 热节流是常态而非异常，需在SLA中预留性能波动范围")
print(f"5. 建立NPU故障的监控告警体系：崩溃率、降级率、延迟P99")
print(f"6. 每次SDK升级后必须重新验证所有模型的精度和性能")

---
## 5.1.14 总结与最佳实践

### NPU适配的核心原则

1. **理解硬件**：深入了解目标NPU的架构（MAC阵列、SRAM容量、精度支持），才能做出正确的优化决策
2. **算子分解优先**：将不支持的算子分解为基础算子组合，比CPU回退更高效
3. **量化是第一优化**：NPU推理的瓶颈是内存带宽，量化直接减少数据搬运量
4. **Profile驱动优化**：先Profile定位瓶颈，再针对性优化，避免盲目调优
5. **异构协同**：CPU和NPU各有所长，合理分工比强求全NPU化更实际

### NPU适配Checklist

- [ ] 确认目标NPU型号和SDK版本
- [ ] 检查模型算子兼容性，制定分解/替换策略
- [ ] 根据内存预算选择量化配置
- [ ] 规划SRAM使用和Tiling策略
- [ ] 处理动态shape（padding/多shape编译）
- [ ] 理解NPU编译器的融合/内存规划/调度策略
- [ ] 执行NPU编译，生成优化二进制
- [ ] 逐层精度验证，定位敏感层
- [ ] 性能Profile，识别瓶颈
- [ ] 异构调度优化，减少CPU↔NPU切换
- [ ] 选择跨平台抽象层（如需多平台部署）
- [ ] 实现NPU→CPU降级策略和错误处理
- [ ] 功耗和热管理测试
- [ ] 长时间稳定性测试（降频、内存泄漏）
- [ ] OTA更新兼容性验证和灰度发布流程